# Workbook 5 — Universality: the machines, they just want to learn

**Exercises · Mysteries of Deep Learning, Iliad Intensive 2026**

Different architectures trained on similar data often end up with similar loss curves, similar internal geometry and even similar hidden quirks. This workbook asks what "similar" can mean, derives one model in which the minima favoured by a small-learning-rate approximation to SGD share a representation, and collects readings on representation geometry, hidden signals in data and scaling curves.

Route (sections are largely independent; §2 gives vocabulary used in §3–4):

- [1. Another bitter lesson: architecture doesn't matter](#wb5-section-1). Short; one reading.
- [2. Same up to what?](#wb5-section-2). Similarity indices and their invariance groups. Exercises: the groups of CKA and CCA; permutation and linear symmetries of networks.
- [3. The Platonic Representation Hypothesis](#wb5-section-3). Two core readings, and Ziyin and Chuang's perfect PRH for deep linear networks derived step by step in exercises.
- [4. The Linear Representation Hypothesis](#wb5-section-4). Two readings; a short exercise on why "orthogonal concepts" needs a chosen inner product.
- [5. Superposition, adversarial attacks, error correction](#wb5-section-5). Core reading; exercises on near-orthogonal packing and the two-feature superposition boundary, with an interactive figure.
- [6. Hidden signals: low-rank logits and subliminal learning](#wb5-section-6). Four readings, one contested. Exercises: logit rank, the shared-initialization argument, and logit-linear selection.
- [7. Scaling laws: a hundred unnecessary explanations](#wb5-section-7). Exercises deriving power-law learning curves from a data spectrum and from discrete skills, including ridge saturation; interactive figure.

**How to use this notebook.** Pick what interests you; nobody is expected to do everything. Exercises are derivations with hints. Worked solutions are in the Solutions version of this notebook, which everyone may use. Each reading opens with a sentence or two on what it shows, so you can decide whether to open it; *core* marks where to start.

<a id="wb5-section-1"></a>

## 1. Another bitter lesson: architecture doesn't matter

The claim to examine: among architectures that are expressive and trainable enough, data and compute determine most of what is learned. The rest of the workbook tests it on internal representations (§2–4), on shared quirks (§5–6) and on loss curves (§7).

**[Scaling laws for neural language models](https://arxiv.org/abs/2001.08361)**, Kaplan et al. (2020) · *optional*

For transformer language models, loss depends strongly on parameter count, data and compute, and only weakly on shape (depth versus width) over a wide range. It also compares LSTMs, which match transformers on early tokens of the context but not on later ones. WB3 §5 uses the same paper for compute allocation.

<a id="wb5-section-2"></a>

## 2. Same up to what?

A claim that two networks "learn the same representation" means something only once you say which transformations count as cosmetic. Each similarity measure fixes such a group.

**[Similarity of neural network representations revisited](https://arxiv.org/abs/1905.00414)**, Kornblith, Norouzi, Lee & Hinton (ICML 2019) · *core*

Shows that similarity indices invariant to all invertible linear maps, CCA included, cannot distinguish representations once the width reaches the number of data points. Proposes comparing representations with centered kernel alignment (CKA), which identifies corresponding layers across networks trained from different initializations. Exercise 2.1 derives the invariance groups.

**[Revisiting model stitching to compare neural representations](https://arxiv.org/abs/2106.07682)**, Bansal, Nakkiran & Barak (NeurIPS 2021) · *optional*

Model stitching joins the lower layers of one network to the upper layers of another through a trainable (usually affine) layer. The paper finds that good networks of the same architecture trained in very different ways, for example supervised and self-supervised, stitch with little loss, and that representations trained with more data, width or time can improve weaker models. Stitching tests whether a representation is usable downstream, not whether its geometry matches.

### 2.A. Setup

Take $n$ inputs. Stack a layer's activations from two networks as the rows of $X\in\mathbb R^{n\times p_1}$ and $Y\in\mathbb R^{n\times p_2}$, and center the columns ($X\leftarrow HX$ with $H=I-\tfrac1n\mathbf 1\mathbf 1^\top$). Write $K=XX^\top$ and $L=YY^\top$ for the centered Gram matrices, whose entries are inner products between inputs. A similarity index $s$ is *invariant to a group* $G$ if $s(XA,YB)=s(X,Y)$ for all $A,B\in G$.

- **Linear CKA** (defined when $K,L\neq0$):
$$\mathrm{CKA}(X,Y)=\frac{\langle K,L\rangle_F}{\|K\|_F\,\|L\|_F}.$$
- **CCA:** the canonical correlations of $X$ and $Y$ are the cosines of the principal angles between their column spaces $\mathrm{col}(X),\mathrm{col}(Y)\subseteq\mathbb R^n$. So CCA depends on $X$ and $Y$ only through these subspaces.
- **Mutual $k$-nearest neighbours**, the index Huh et al. use in §3, asks only whether each input's $k$ nearest neighbours agree between the two representations.

**Exercise 2.1.** The invariance groups of CKA and CCA.

(a) Show that $\langle K,L\rangle_F=\|Y^\top X\|_F^2$. Then show that $\mathrm{CKA}(XQ,Y)=\mathrm{CKA}(X,Y)$ for orthogonal $Q$ and $\mathrm{CKA}(cX,Y)=\mathrm{CKA}(X,Y)$ for $c\neq0$. Finally, find an invertible $A$ and data $X$ with $\mathrm{CKA}(XA,X)<1$.

(b) Show that $0\le\mathrm{CKA}\le1$, with $\mathrm{CKA}=1$ if and only if $K=cL$ for some $c>0$.

(c) Suppose $K=cL$, and let $x_i,y_i$ be the rows of $X,Y$. Show that there is a linear isometry $R$ from $\mathrm{span}\{x_i\}$ onto $\mathrm{span}\{y_i\}$ with $y_i=c^{-1/2}Rx_i$ for every $i$. When $p_1\neq p_2$, $R$ is an isometry between these occupied subspaces, not an orthogonal matrix on the whole space.

(d) The rows in (c) are centered. What extra freedom does that leave between the *uncentered* activations when $\mathrm{CKA}=1$?

(e) Show that CCA is invariant to all invertible $A,B$. Then show that if the centered $X$ has rank $n-1$ (possible only if $p_1\ge n-1$), $\mathrm{col}(X)$ is the whole space $\mathbf 1^\perp$ of centered vectors. Conclude that any two such representations have all canonical correlations equal to 1.

<details><summary>Hints</summary>

(a) $\langle K,L\rangle_F=\mathrm{Tr}(XX^\top YY^\top)$; use cyclicity of the trace. For the counterexample, try four points in $\mathbb R^2$ at $(\pm1,0)$ and $(0,\pm1)$ with $A=\mathrm{diag}(2,1)$.
(b) Cauchy–Schwarz for the Frobenius inner product, and $\mathrm{Tr}(KL)\ge0$ for positive semidefinite $K,L$.
(c) Define $R(\sum_ia_ix_i)=c^{1/2}\sum_ia_iy_i$ and check that it is well defined.
(e) Each column of a centered $X$ is orthogonal to $\mathbf 1$.
</details>

**Exercise 2.2.** Parameter symmetries: many settings, one function.

(a) Take a one-hidden-layer network $f(x)=W_2\,\sigma(W_1x+b_1)+b_2$ with $\sigma$ applied elementwise. For a permutation matrix $P$, show that $(PW_1,Pb_1,W_2P^\top)$ computes the same $f$. If $\sigma$ is ReLU, show the same for $(DW_1,Db_1,W_2D^{-1})$ with $D$ diagonal with positive entries. For a generic MLP with hidden widths $n_1,\dots,n_{L-1}$, how many parameter settings computing the same function does permutation alone give?

(b) Take a deep linear network $f(x)=W_D\cdots W_1x$. For invertible $T$, show that $W_i\to TW_i$, $W_{i+1}\to W_{i+1}T^{-1}$ leaves $f$ unchanged and maps the layer-$i$ representation $h$ to $Th$. Suppose the representations $\{h(x)\}$ over the data span $\mathbb R^{d_i}$. Show that the new Gram matrix is a positive multiple of the old one, $h(x)^\top T^\top Th(x')=c\,h(x)^\top h(x')$ for all pairs, if and only if $T^\top T=cI$, that is, $T$ is a scaled orthogonal matrix.

(c) Under which of §2.A's indices are the layer-$i$ representations before and after a general $T$ "the same"? (CKA centers its inputs, so check which spanning assumption (b) needs for it.)

<details><summary>Hint</summary>

(a) An elementwise $\sigma$ commutes with permutations, and $\mathrm{ReLU}(Dz)=D\,\mathrm{ReLU}(z)$ for positive diagonal $D$. (b) A bilinear form that vanishes on a spanning set vanishes everywhere.
</details>

<a id="wb5-section-3"></a>

## 3. The Platonic Representation Hypothesis

Huh et al. propose that the representations of large models are converging toward a shared one. Ziyin and Chuang prove a "perfect" version for deep linear networks, for the minima selected by a small-learning-rate approximation to SGD. They do not prove that SGD's trajectory reaches those minima.

**[The Platonic representation hypothesis](https://arxiv.org/abs/2405.07987)**, Huh, Cheung, Wang & Isola (ICML 2024) · *core*

Reports that representations in different models, architectures and even modalities become more aligned as the models become more capable, measured mainly by a mutual nearest-neighbour index. Hypothesizes convergence toward a shared statistical model of the world, driven by three pressures: multitask scaling, capacity and a simplicity bias. It discusses limitations, including information specific to one modality that cannot converge.

**[Proof of a perfect Platonic representation hypothesis](https://arxiv.org/abs/2507.01098)**, Ziyin & Chuang (2025) · *core*

Two deep linear networks of any depths and of hidden widths at least the rank of the target map, between arbitrary fixed invertible maps and trained on different invertible views of the same noisy-label data, have every pair of hidden layers aligned up to rotation and scale. This holds at the minima selected by SGD's small-learning-rate modified objective, in which a gradient-noise term breaks ties among global minima. It does not prove that SGD converges to these minima (its Remark leaves that to future work); Exercises 3.1–3.4 follow its proof and part of its list of ways the result breaks.

### 3.A. Setup

**Model.** An *embedded deep linear network* computes
$$f(x)=M^{O}\,W_D\cdots W_1\,M^{I}\,x,\qquad D\ge2 .$$
The $W_j$ are trained. $M^I$ and $M^O$ are fixed invertible matrices that stand in for the rest of an architecture. The hidden widths are at least $r=\mathrm{rank}\,V^*\ge1$ (defined below).

**Data.** Underlying inputs $x$ have second moment $\Sigma_x=\mathbb E[xx^\top]\succ0$. If $\Sigma_x$ is singular, restrict everything below to its range and read $\Sigma_x^{-1/2}$ as a pseudoinverse square root. Labels are $y=V^*x+\epsilon$, with $\epsilon$ zero-mean, independent of $x$, and of covariance $\Sigma_\epsilon\succ0$. Network $A$ sees $x^A=Z_Ax$ and network $B$ sees $x^B=Z_Bx$, with $Z_A,Z_B$ invertible: two "modalities" that carry the same information. Both are trained on the same labels with per-sample loss
$$\ell=\tfrac12\|f(x^A)-y\|^2,\qquad L=\mathbb E\,\ell .$$

**Representations.** For $1\le i\le D-1$, hidden layer $i$ of network $A$ is $h^i_A(x)=W_i\cdots W_1M^I_AZ_Ax$. Layers $h_A$ and $h_B$ are **perfectly aligned** if, for some $c_0>0$,
$$h_A(x)^\top h_A(x')=c_0\,h_B(x)^\top h_B(x')\qquad\text{for all }x,x' .$$
These representations are linear in $x$ with no bias, so the condition also holds after centering, and it gives linear $\mathrm{CKA}=1$ on any data set whose centered representations are nonzero. By Exercise 2.1(c) it means $h_B=c_0^{-1/2}R\,h_A$ with $R$ an isometry between the occupied subspaces.

**Selection.** Backward-error analyses of gradient descent and SGD show that small-step training approximately follows gradient flow on a modified objective with an added gradient-norm penalty proportional to the learning rate $\eta$. Ziyin and Chuang write it as $\mathbb E[\ell]+\eta\,S$, with
$$S(\theta)=\mathbb E\,\|\nabla_\theta\ell\|^2$$
(their equation (12)), which they call an "entropy". At a global minimum of $L$ the mean gradient vanishes, so $S$ is the total variance of the per-sample gradients, that is, SGD's gradient noise at the solution. Ziyin and Chuang take the $\eta\to0^+$ limit of the modified objective's minimizers: minimize $S$ among the global minima of $L$ (their equation (13)). Those minimizers are what the theorem characterizes. Gradient flow, which takes $\eta\to0$ along the whole trajectory, never sees $S$.

**Exercise 3.1.** Global minima, and why being one is not enough.

(a) Show that $L\ge\tfrac12\mathrm{Tr}\,\Sigma_\epsilon$, with equality if and only if $M^OW_D\cdots W_1M^IZ_A=V^*$, that is,
$$W_D\cdots W_1=(M^O)^{-1}\,V^*\,Z_A^{-1}\,(M^I)^{-1}.$$
Where did you use $\Sigma_x\succ0$?

(b) Take layer $i$ of width $r\ge2$. Using Exercise 2.2(b), show that the set of global minima contains points whose layer-$i$ representations are not perfectly aligned with each other. So any alignment has to come from the selection rule.

<details><summary>Hints</summary>

(a) Write $E=M^OW_D\cdots W_1M^IZ_A$ for the end-to-end map from $x$, so $f(x^A)-y=(E-V^*)x-\epsilon$. Expand the square; the cross term vanishes by independence and $\mathbb E\epsilon=0$. Matrices do not commute: solve for $W_D\cdots W_1$ by multiplying by inverses on the correct sides.
(b) At width $r$, the map $x\mapsto h^i(x)$ has rank $r$ because the end-to-end map does.
</details>

**Exercise 3.2.** The balance condition.

Fix $1\le i\le D-1$. Write $K=W_{i+1}W_i$ and, per sample, $G=\nabla_K\ell$.

(a) Show that $\nabla_{W_i}\ell=W_{i+1}^\top G$ and $\nabla_{W_{i+1}}\ell=G\,W_i^\top$.

(b) For a symmetric matrix $T$ and $\lambda\in\mathbb R$, consider the move
$$W_i\to e^{\lambda T}W_i,\qquad W_{i+1}\to W_{i+1}e^{-\lambda T}.$$
It keeps $K$ fixed, and with it $f$, $\ell$, $G$ and the gradients of all other layers. Show that $S$ changes only through
$$S_i(\lambda)=\mathrm{Tr}\big[e^{-2\lambda T}A\big]+\mathrm{Tr}\big[e^{2\lambda T}B\big],\qquad A=\mathbb E\big[W_{i+1}^\top GG^\top W_{i+1}\big],\quad B=\mathbb E\big[W_iG^\top GW_i^\top\big].$$

(c) Show that at any minimizer of $S$ among the global minima of $L$,
$$W_{i+1}^\top\,\mathbb E[GG^\top]\,W_{i+1}=W_i\,\mathbb E[G^\top G]\,W_i^\top . \tag{$*$}$$

(d) Show that $S_i$ is convex in $\lambda$, and that $S$ is constant along $W_i\to OW_i$, $W_{i+1}\to W_{i+1}O^\top$ for orthogonal $O$. Conclude that the minimizer on a symmetry orbit need not be unique. The derivation uses $(*)$ only as a necessary condition.

<details><summary>Hints</summary>

(a) $\ell$ depends on $W_i,W_{i+1}$ only through $K$, and $dK=dW_{i+1}\,W_i+W_{i+1}\,dW_i$.
(c) The moves of (b) stay among the global minima, so $\lambda=0$ must be a stationary point of $S_i$. For the last step, choose $T=B-A$.
(d) For PSD $A$, $\mathrm{Tr}[Te^{-2\lambda T}TA]=\|e^{-\lambda T}TA^{1/2}\|_F^2$.
</details>

**Exercise 3.3.** Solve the balance condition and read off the representation.

(a) At a global minimum, show that $G=g\,h^\top$, where $h=Nx$ is the input to $W_i$ and $g=-\bar M^\top\epsilon$ is the gradient arriving at the output of $W_{i+1}$, with
$$N=W_{i-1}\cdots W_1M^IZ_A\ (\text{$N=M^IZ_A$ if $i=1$}),\qquad \bar M=M^OW_D\cdots W_{i+2}\ (\text{$\bar M=M^O$ if $i=D-1$}).$$
Deduce, using the independence of $\epsilon$ and $x$, that $(*)$ becomes
$$\mathbb E\|h\|^2\;W_{i+1}^\top\bar M^\top\Sigma_\epsilon\bar M\,W_{i+1}=\mathbb E\|g\|^2\;W_iN\Sigma_xN^\top W_i^\top .$$

(b) Define $\widetilde W_i=W_iN\Sigma_x^{1/2}$ and $\widetilde W_{i+1}=\Sigma_\epsilon^{1/2}\bar MW_{i+1}$. Show that the global-minimum condition and the balance condition become
$$\widetilde W_{i+1}\widetilde W_i=\bar V:=\Sigma_\epsilon^{1/2}V^*\Sigma_x^{1/2},\qquad \widetilde W_{i+1}^\top\widetilde W_{i+1}=c\,\widetilde W_i\widetilde W_i^\top,\qquad c=\frac{\mathbb E\|g\|^2}{\mathbb E\|h\|^2}>0 .$$
Neither equation mentions $M^I$, $M^O$ or $Z_A$. The scalar $c$ is set by the other layers of the same solution, so it can differ between layers and networks (write $c_{A,i}$ when it matters). Below, it is treated as a given positive number.

(c) Let $\bar V=PSQ^\top$ be a compact SVD, with $S$ an $r\times r$ positive diagonal matrix. Show that for layer width $d\ge r$, every solution has the form
$$\widetilde W_i=c^{-1/4}\,R\,S^{1/2}Q^\top,\qquad \widetilde W_{i+1}=c^{1/4}\,P\,S^{1/2}R^\top,\qquad R\in\mathbb R^{d\times r},\ R^\top R=I_r .$$

(d) Show that
$$h^i(x)^\top h^i(x')=c^{-1/2}\,x^\top\Sigma_x^{-1/2}\,QSQ^\top\,\Sigma_x^{-1/2}x' .$$
Conclude that every pair of hidden layers, within network $A$ and across $A$ and $B$, is perfectly aligned, and that $h_B^j=(c_A/c_B)^{1/4}R_BR_A^\top h_A^i$, where $R_BR_A^\top$ is an isometry from the occupied subspace of $h^i_A$ onto that of $h^j_B$.

(e) The shared representation depends on the labels. Network $A$ is trained on $y$ and network $B$ on $\Phi y$, with $\Phi$ invertible, and everything else is as before. Show that the two are perfectly aligned if $\Phi$ is a scaled orthogonal matrix. Then give a $\Phi$ for which they are not.

<details><summary>Hints</summary>

(a) At a global minimum, $f(x^A)-y=-\epsilon$, and $f=\bar M\,K\,h$. Then $GG^\top=\|h\|^2gg^\top$ and $G^\top G=\|g\|^2hh^\top$.
(b) $\bar MW_{i+1}W_iN$ is the end-to-end map of Exercise 3.1(a).
(c) Write $\mathcal A=\widetilde W_i$ and $\mathcal B=\widetilde W_{i+1}$. Compute $\bar V^\top\bar V=\mathcal A^\top\mathcal B^\top\mathcal B\mathcal A$ using the balance equation, and take the PSD square root. Two matrices with the same Gram matrix $\mathcal A^\top\mathcal A=\mathcal C^\top\mathcal C$, where $\mathcal C$ has full row rank, satisfy $\mathcal A=R\,\mathcal C$ with $R^\top R=I$. For $\mathcal B$, show that it vanishes on the orthogonal complement of $\mathrm{range}(R)$.
(d) $h^i=W_iNx=\widetilde W_i\Sigma_x^{-1/2}x$, and $QSQ^\top=(\bar V^\top\bar V)^{1/2}$.
(e) Track how $V^*$ and $\Sigma_\epsilon$ change, then use $QSQ^\top=(\bar V^\top\bar V)^{1/2}$.
</details>

**Exercise 3.4.** What selects nothing, and what selects something else.

(a) Without label noise ($\Sigma_\epsilon=0$), show that $S=0$ at every global minimum.

(b) Under full-batch gradient flow on $L$, $\dot W_j=-\nabla_{W_j}L$. Show that $W_{i+1}^\top W_{i+1}-W_iW_i^\top$ is conserved, so any limit point keeps an imprint of the initialization.

(c) With weight decay, $L_\gamma=L+\tfrac\gamma2\sum_j\|W_j\|_F^2$ with $\gamma>0$. Show that every stationary point of $L_\gamma$ satisfies *norm balance*, $W_iW_i^\top=W_{i+1}^\top W_{i+1}$. Now take $D=2$, $M^I=M^O=I$, $V^*=\Sigma_x=I_2$ and $Z_A=\mathrm{diag}(z_1,z_2)$ with $z_k>0$, and look at global minima of $L$ that satisfy norm balance (the $\gamma\to0^+$ picture). Show that the hidden Gram matrix is $x^\top Z_Ax'$, and compare it with a network $B$ that has $Z_B=I$. For comparison, find what Exercise 3.3 gives when $\Sigma_\epsilon=\sigma^2I$ with $\sigma>0$.

(d) In the model of Exercises 3.1–3.3, which of Huh et al.'s three pressures (multitask scaling, capacity, simplicity bias) does the selection use?

<details><summary>Hints</summary>

(b) Use Exercise 3.2(a) with $\bar G=\mathbb E\,G$ and differentiate both products.
(c) $L$ is unchanged by the moves of Exercise 3.2(b); differentiate the penalty at $\lambda=0$. For the example, mimic Exercise 3.3(c) with $c=1$ and no whitening: $(W_2W_1)^\top(W_2W_1)=(W_1^\top W_1)^2$.
</details>

<a id="wb5-section-4"></a>

## 4. The Linear Representation Hypothesis

The hypothesis says that high-level concepts are directions in a model's representation space. One reading makes it precise and argues that its geometry needs a chosen inner product; the other finds features that are irreducibly multi-dimensional.

**[The linear representation hypothesis and the geometry of large language models](https://arxiv.org/abs/2311.03658)**, Park, Choe & Veitch (ICML 2024) · *core*

Formalizes the hypothesis with counterfactual pairs, in the unembedding space (linked to linear probing) and in the embedding space (linked to steering). Shows that a non-Euclidean 'causal inner product', under which causally separable concepts are orthogonal, unifies the two; LLaMA-2 experiments find such representations and show that the choice of inner product matters. Exercise 4.1 works through the non-identifiability that makes the inner product a choice.

**[Not all language model features are one-dimensionally linear](https://arxiv.org/abs/2405.14860)**, Engels, Michaud, Liao, Gurnee & Tegmark (ICLR 2025) · *counterpoint*

Uses sparse autoencoders on GPT-2 and Mistral 7B to find features that cannot be split into independent or non-co-occurring lower-dimensional parts, most clearly circles for days of the week and months of the year. Intervention experiments show that models use these circles for modular arithmetic on days and months.

### 4.A. Setup

A language model's next-token logits are $\lambda(x)^\top\gamma(y)$, where $\lambda(x)\in\mathbb R^d$ is the final hidden state for context $x$ and $\gamma(y)$ is the unembedding vector of token $y$. Concept directions are differences such as $\gamma(y_1)-\gamma(y_0)$ over counterfactual pairs. Park et al. estimate their causal inner product as
$$\langle a,b\rangle_{\mathrm C}=a^\top\,\mathrm{Cov}(\gamma)^{-1}\,b,$$
with the covariance taken over the unembedding vectors of the vocabulary (assume it is invertible).

**Exercise 4.1.** Which inner product?

(a) Show that for any invertible $A$, the change $\lambda\to A\lambda$, $\gamma\to A^{-\top}\gamma$ leaves every logit, and hence the model's behaviour, unchanged.

(b) Give two unembedding differences that are orthogonal in the Euclidean inner product before such a change and not after it.

(c) Show that $\langle a,b\rangle_{\mathrm C}$, recomputed in the new coordinates (with $a,b$ transformed as unembedding differences and the covariance recomputed), equals its old value.

(d) Show that in the coordinates $g(y)=\mathrm{Cov}(\gamma)^{-1/2}\big(\gamma(y)-\mathbb E\gamma\big)$, $\langle\cdot,\cdot\rangle_{\mathrm C}$ on differences is the Euclidean inner product.

<details><summary>Hint</summary>

(c) If $\gamma'=A^{-\top}\gamma$, then $\mathrm{Cov}(\gamma')=A^{-\top}\mathrm{Cov}(\gamma)A^{-1}$.
</details>

<a id="wb5-section-5"></a>

## 5. Superposition, adversarial attacks, error correction

High-dimensional space holds many almost-orthogonal directions, so a network can store more sparse features than it has dimensions, at the price of interference. Interference is one proposed account of adversarial vulnerability; WB6 §5 has a competing one ("Adversarial examples are not bugs, they are features"). Storing a feature redundantly is a candidate route to error correction, for which this workbook has no reading.

**[Toy models of superposition](https://transformer-circuits.pub/2022/toy_model/index.html)**, Elhage et al. (Anthropic, 2022; also arXiv 2209.10652) · *core*

In a toy ReLU autoencoder, networks store more sparse features than they have dimensions, which offers an explanation of polysemantic neurons. Whether they do so is a phase change in sparsity and importance, the stored features form regular polytopes, and in these toy models superposition goes together with adversarial vulnerability. The section 'Superposition as a Phase Change' has the two-feature phase diagram that Exercise 5.2 derives.

**Exercise 5.1.** Packing almost-orthogonal directions.

Let $u,v$ be independent, uniformly random unit vectors in $\mathbb R^m$.

(a) Show that $\mathbb E\langle u,v\rangle=0$ and $\mathbb E\langle u,v\rangle^2=1/m$.

(b) Take as given the concentration bound $\Pr(|\langle u,v\rangle|>\varepsilon)\le2e^{-m\varepsilon^2/2}$. Show that if $N^2e^{-m\varepsilon^2/2}<1$, then $N$ independent random unit vectors have all pairwise $|\cos|\le\varepsilon$ with positive probability. Conclude that $\mathbb R^m$ contains about $e^{m\varepsilon^2/4}$ such directions.

<details><summary>Hint</summary>

(a) Condition on $v$ and use $\mathbb E[uu^\top]=I/m$ (isotropy, and the trace is 1). (b) Union bound over pairs.
</details>

### 5.A. The toy model

There are $n$ features $x\in\mathbb R^n$, independent, each zero with probability $S$ (the sparsity) and otherwise uniform on $[0,1]$. They are compressed into $m<n$ dimensions and read back:
$$h=Wx\in\mathbb R^m,\qquad x'=\mathrm{ReLU}(W^\top Wx+b),\qquad L=\mathbb E\sum_iI_i\,(x_i-x_i')^2,$$
with $W\in\mathbb R^{m\times n}$ and per-feature importances $I_i$. Column $W_{:,i}$ is where feature $i$ is stored.

**Exercise 5.2.** The smallest case: $n=2$ features in $m=1$ dimension, importances $(1,r)$ with $0<r\le1$, and density $p=1-S\in(0,1]$.

(a) *Dedicate the dimension to feature 1:* $W=[1,0]$, $b=(0,b_2)$ with $b_2\ge0$. Show that feature 1 is reconstructed exactly, that the best $b_2$ is $\mathbb Ex_2=p/2$, and that the loss is $r\,(p/3-p^2/4)$.

(b) *Antipodal superposition:* $W=[1,-1]$, $b=0$. Show that the loss is $(1+r)\,p^2/6$.

(c) Show that the antipodal solution has lower loss exactly when $p<p^*$, where
$$p^*=\frac{r/3}{(1+r)/6+r/4}.$$
Evaluate $p^*$ at $r=1$ and as $r\to0$.

<details><summary>Hints</summary>

$\mathbb Ex_2=p/2$ and $\mathbb Ex_2^2=p/3$. For (b), split into cases by which features are active; if both are, compare $x_1'$ with $\min(x_1,x_2)$. The minimum of two independent uniforms on $[0,1]$ has density $2(1-t)$.
</details>

▶ **Interactive figure:** [`superposition.html`](assets/interactive/superposition.html). Panel 1 is the $n=2$, $m=1$ phase diagram over density and relative importance: the dashed boundary is Exercise 5.2's closed form, the solid one optimizes weights and biases, and dots are trained models. Panel 2 trains $n=5$ or $6$ features in $m=2$ dimensions in your browser; as sparsity increases, the features go from dedicated dimensions to antipodal pairs to a regular polygon.

<a id="wb5-section-6"></a>

## 6. Hidden signals: low-rank logits and subliminal learning

Readings on structure shared across language models' outputs, and on traits passed between models through data that never mentions them, with derivations of the simplest version of each mechanism.

### 6.A. Low-rank logits

**[Sequences of logits reveal the low rank structure of language models](https://arxiv.org/abs/2510.24966)**, Golowich, Liu & Shetty (2025) · *core*

Matrices of log-probabilities of multi-token continuations given prompts are approximately low-rank across many modern language models, and the linear relations among prompts are consistent across models. The authors use this to generate a response to a target prompt from the model's outputs on unrelated or nonsensical prompts. The theory part (a simple model with exact low logit rank, and learning guarantees) is optional.

**Exercise 6.1.** A model with hidden size $d$ has next-token log-probabilities
$$\log p(y\mid h)=\phi(h)^\top\psi(y)-\log\sum_{y'}e^{\phi(h)^\top\psi(y')},$$
with $\phi(h)\in\mathbb R^d$ the final hidden state for history $h$.

(a) Show that the matrix with rows indexed by histories, columns by next tokens, and entries $\log p(y\mid h)$ has rank at most $d+1$.

(b) For a multi-token future $f=(f_1,\dots,f_k)$, $\log\Pr[f\mid h]=\sum_{t=1}^k\log p(f_t\mid h,f_{<t})$. Explain why (a) does not bound the rank of the matrix with entries $\log\Pr[f\mid h]$ when $k\ge2$.

### 6.B. Subliminal learning

**[Subliminal learning: language models transmit behavioral traits via hidden signals in data](https://arxiv.org/abs/2507.14805)**, Cloud, Le, Chua, Betley, Sztyber-Betley, Hilton, Marks & Evans (2025) · *core*

A student fine-tuned on number sequences, code or chain-of-thought generated by a teacher with a trait (a preferred animal, or misalignment) acquires the trait even when references to it are filtered out. The effect is reported only when teacher and student share a base model or initialization. Exercise 6.2 derives the paper's argument that a shared initialization suffices for one small step, which an MNIST experiment also illustrates. Short summary: [Subliminal learning (Anthropic Alignment Science blog)](https://alignment.anthropic.com/2025/subliminal-learning/).

**[Subliminal learning is a LoRA artifact](https://arxiv.org/abs/2606.00831)**, Nief, Fu, Muchane & Holtzman (2026) · *contested*

Reports that subliminal learning in the number-sequence setting depends non-monotonically on LoRA rank, disappears under full fine-tuning, and is tied to tokens seen in both training and evaluation, such as the system prompt and chat template. Its arXiv listing states that follow-up work showed some of its main claims to be incorrect and that a substantial revision is in preparation, so it does not settle the question either way. Cloud et al.'s MNIST experiment uses no LoRA.

**Exercise 6.2.** Why a shared initialization suffices for one small step.

Let $f_\theta(x)\in\mathbb R^k$ be a model's outputs, and $\ell(u,t)$ a smooth loss of outputs $u$ against targets $t$ that, for each $t$, is minimized over $u$ at $u=t$, so $\nabla_u\ell(t,t)=0$. Examples are half squared error, and the cross-entropy $\mathrm{CE}(\mathrm{softmax}(t),\mathrm{softmax}(u))$ of logits $u$ against the teacher's soft targets. Write $H(t)=\nabla^2_{uu}\ell(u,t)\big|_{u=t}\succeq0$. The teacher is $\theta_T=\theta_0+\varepsilon\Delta$ with small $\varepsilon>0$. The student starts at $\theta_0$ and takes one gradient step of size $\eta>0$ on $L_S(\theta)=\mathbb E_x\,\ell\big(f_\theta(x),f_{\theta_T}(x)\big)$, for an input distribution with finite expectations permitting the Taylor expansions below to pass under the expectation. Assume $f$ is smooth in its parameters; a finite input distribution suffices for this interchange.

(a) Show that $\Delta\theta_S=\eta\,\varepsilon\,\mathbb E_x\big[J^\top HJ\big]\Delta+O(\varepsilon^2)$, with $J=\partial f_\theta(x)/\partial\theta$ at $\theta_0$ and $H=H(f_{\theta_0}(x))$.

(b) Deduce that $\Delta\theta_S\cdot\Delta\ge0$ to leading order in $\varepsilon$.

(c) Suppose the teacher was made by a small gradient step on its own loss $L_T$, so $\Delta=-\nabla L_T(\theta_0)$. Show that the student's step does not increase $L_T$ to first order.

(d) Where did the argument use the shared initialization? What changes if the student starts at a different $\theta_0'$, or has a different architecture?

<details><summary>Hint</summary>

(a) The student's gradient is $J^\top\nabla_u\ell(u_0,t)$ with $u_0=f_{\theta_0}(x)$ and $t=u_0+\varepsilon J\Delta+O(\varepsilon^2)$. Since $\nabla_u\ell(t,t)=0$ for all $t$, expand $\nabla_u\ell(u_0,t)$ in $u_0-t$ around the point $(t,t)$.
</details>

### 6.C. A log-linear account

**[Subliminal effects in your data: a general mechanism via log-linearity](https://arxiv.org/abs/2602.04863)**, Aden-Ali, Golowich, Liu, Shetty, Moitra & Haghtalab (2026) · *core*

Assuming log-probabilities are approximately bilinear in a system-prompt embedding and a prompt–response embedding shared across models, the paper's Logit-Linear Selection picks subsets of an ordinary preference dataset that make a DPO-trained student behave as if given a target system prompt (preferring an animal, answering in a language absent from the data, adopting a persona). It reports the effect within a model and, more weakly, across architectures, in contrast with Cloud et al. Exercise 6.3 is a one-step version of its mechanism, not its theorem.

**Exercise 6.3.** Logit-linear selection in a bilinear model.

Assume $\log\Pr_M[r\mid s,p]=\psi_M(s)^\top\phi(p,r)-\log Z_M(s,p)$, where $s$ is a system prompt, $p$ a prompt, $r$ a response, and $Z_M$ normalizes over responses. Let $\varnothing$ be the empty system prompt and $u=\psi_T(s)-\psi_T(\varnothing)$ the teacher's trait direction for a target system prompt $s$. A preference dataset has triples $(p_i,r_i^+,r_i^-)$; write $\delta_i=\phi(p_i,r_i^+)-\phi(p_i,r_i^-)$. The selection score, ignoring length normalization, is
$$w_i=\big[\log\Pr_T(r_i^+\mid s,p_i)-\log\Pr_T(r_i^-\mid s,p_i)\big]-\big[\log\Pr_T(r_i^+\mid\varnothing,p_i)-\log\Pr_T(r_i^-\mid\varnothing,p_i)\big].$$

(a) Show that $w_i=u^\top\delta_i$.

(b) The student is in the same bilinear family with the same $\phi$; its parameter is $\psi=\psi_S(\varnothing)$, starting at the reference $\psi_{\rm ref}$. DPO on a selected set $I$ minimizes $\mathcal L(\psi)=-\sum_{i\in I}\log\sigma\big(\beta(\psi-\psi_{\rm ref})^\top\delta_i\big)$. Show that the first gradient step of size $\eta$ is $\Delta\psi=\tfrac{\eta\beta}2\sum_{i\in I}\delta_i$, and that $\Delta\psi^\top u=\tfrac{\eta\beta}2\sum_{i\in I}w_i$, which is positive when $I$ is a set of top-scoring examples with positive scores.

(c) Now suppose the student's embedding is $\phi_S=B\phi$ for an invertible $B$. Recompute the step and the student's change in preference log-ratio on a test pair with teacher-embedding difference $\delta$. Is the change positive on the pair $\delta=\sum_{i\in I}\delta_i$? On a general test pair?

<details><summary>Hint</summary>

(a) The normalizer depends on $(s,p)$ but not on $r$. (b) In DPO the policy-to-reference log-ratio difference is $(\psi-\psi_{\rm ref})^\top\delta_i$ here, and $\frac{d}{dz}\big[-\log\sigma(z)\big]=-(1-\sigma(z))$, which is $-\tfrac12$ at $z=0$.
</details>

<a id="wb5-section-7"></a>

## 7. Scaling laws: a hundred unnecessary explanations

Very different mechanisms produce the same power-law learning curve. The exercises derive one from a power-law data spectrum, with ridge regression, where extra target smoothness eventually stops helping, and one from discrete skills learned in order of frequency.

**[The quantization model of neural scaling](https://arxiv.org/abs/2303.13506)**, Michaud, Liu, Girit & Tegmark (NeurIPS 2023) · *optional*

Proposes that networks learn discrete chunks of skill ('quanta') in order of how often they are used; with Zipfian use frequencies this gives power-law scaling in parameters and data. It uses language-model gradients to decompose model behaviour automatically into candidate quanta.

**[On the asymptotic learning curves of kernel ridge regression under power-law decay](https://arxiv.org/abs/2309.13337)**, Li, Zhang & Lin (2023) · *rabbit hole*

Rigorous learning curves for kernel ridge regression with power-law eigenvalues, showing how the regularization, the target's smoothness and the noise set the rate. The rigorous counterpart of Exercise 7.2's heuristic, including saturation.

### 7.A. Setup

**Spectral model.** Features $x\in\mathbb R^\infty$ are Gaussian with covariance $\mathrm{diag}(\lambda_k)$, $\lambda_k=k^{-\alpha}$, $\alpha>1$. The target is $y=\sum_k\beta_kx_k+\varepsilon$ with noise variance $\sigma^2$. The excess population loss of an estimate $\hat\beta$ is $\sum_k\lambda_k(\hat\beta_k-\beta_k)^2$. The target's signal power in mode $k$ is
$$c_k^2=\lambda_k\beta_k^2=k^{-\alpha'},\qquad\alpha'>1 .$$
A generic target with $\beta_k$ of order 1 has $\alpha'=\alpha$; smoother targets, concentrated on the top modes, have $\alpha'>\alpha$. The language of spectral filters is WB2 §4's.

**Quanta model.** A task consists of discrete skills $k=1,2,\dots$, used with frequencies $p_k\propto k^{-(1+a)}$, $a>0$. A model that has learned the $N$ most frequent skills has loss proportional to the frequency mass of the rest.

**Exercise 7.1.** Two routes to one curve.

(a) Show that $\sum_{k>K}k^{-s}\sim K^{1-s}/(s-1)$ for $s>1$.

(b) *Heuristic for ridge.* Suppose the estimator learns modes $1,\dots,K$ with $K\le n$ and nothing beyond. Each learned mode costs about $\sigma^2/n$: its coefficient has error variance about $\sigma^2/(n\lambda_k)$, weighted by $\lambda_k$ in the loss. Each unlearned mode costs its signal $c_k^2$. So
$$E(K)\approx\sum_{k>K}c_k^2+\frac{\sigma^2K}n .$$
Show that $E\sim n^{-(\alpha'-1)}$ without noise, and that $E\sim n^{-(\alpha'-1)/\alpha'}$ with noise and the best $K$.

(c) Show that the quanta model's loss after learning $N$ skills scales as $N^{-a}$. If $N$ grows as a power of parameters or data, what form does the loss curve take?

**Exercise 7.2.** Where the heuristic fails: saturation.

Ridge shrinks mode $k$ by the filter $\lambda_k/(\lambda_k+\kappa)$, so it also biases the modes it "learns". In the large-$n$ description of ridge, the effective regularization $\kappa$ sits around the eigenvalue of the last resolved mode. Without noise, take $\kappa\approx\lambda_n=n^{-\alpha}$. The bias from shrinking the head is
$$B_{\rm head}=\sum_{k<n}c_k^2\Big(\frac{\kappa}{\lambda_k+\kappa}\Big)^2 .$$

(a) Show that $B_{\rm head}\approx n^{-2\alpha}\sum_{k<n}k^{2\alpha-\alpha'}$ up to constants, and that modes with $k$ comparable to $n$ contribute at most the order of the tail, $n^{1-\alpha'}$.

(b) Evaluate the sum when $\alpha'-1<2\alpha$, when $\alpha'-1>2\alpha$, and at the boundary $\alpha'-1=2\alpha$. Conclude that the noiseless rate is $n^{-\min(\alpha'-1,\,2\alpha)}$ away from the boundary, and find the rate at the boundary.

(c) With noise and optimally tuned ridge, repeat Exercise 7.1(b) with $s=\min(\alpha'-1,2\alpha)$ in place of $\alpha'-1$, taking the cutoff $K$ as the free variable. Show that $E\sim n^{-s/(s+1)}$ away from the boundary, and that this reduces to Exercise 7.1(b) when $s=\alpha'-1$.

<details><summary>Hint</summary>

(a) For $k\le n/2$, $\lambda_k\ge2^\alpha\kappa$, so $\kappa/(\lambda_k+\kappa)$ is within a constant of $\kappa/\lambda_k=n^{-\alpha}k^\alpha$. For $n/2<k<n$ the filter factor is at most 1. (b) $\sum_{k<n}k^q$ grows like $n^{q+1}$ for $q>-1$, like $\log n$ for $q=-1$, and converges for $q<-1$.
</details>

▶ **Interactive figure:** [`spectral_filters.html`](assets/interactive/spectral_filters.html). Use panel B. It plots the deterministic-equivalent ridge risk (an asymptotic approximation) and Exercise 7.1's heuristic against $n$, with the fitted slope, plus Monte Carlo ridge points at the default $\alpha,\alpha'$ (noise off, or $\sigma^2=10^{-2}$). Turn noise on and off, and push $\alpha'$ past $2\alpha+1$ to see saturation, where the heuristic's slope is wrong.

## Related workbooks and further study

- Ziyin, Xu and Chuang, [Neural thermodynamics: entropic forces in deep and universal representation learning](https://arxiv.org/abs/2505.12387) (NeurIPS 2025). The full paper behind §3. It also uses the same entropic forces to account for both sharpness-seeking and flatness-seeking behaviour of SGD.
- The Training Dynamics materials develop the dynamics behind §3's selection.
- WB6 §3 (model stitching) and WB6 §5 (adversarial examples) are experimental counterparts to §2 and §5.
- WB3 §5 fits the scaling laws whose exponents §7 discusses.